# original excel to schema migration

In [1]:
import pandas as pd

### trtmtEnv from all_consonants_data

In [35]:
acd = pd.read_csv('all_consonants_data.csv')

In [36]:
acd.columns

Index(['Unnamed: 0', 'treatment', 'environment', 'position', 'number',
       'Languages', 'Consonantal +/−', 'Grapheme', 'IPA', 'IPA key',
       'Lateral +/−', 'Manner of articulation', 'Place of articulation',
       'Place value', 'Sibilant +/−', 'Sonority value', 'Voice +/−', 'version',
       'voice', 'place', 'sonority'],
      dtype='object')

In [37]:
trtmtEnv = acd[['treatment', 'environment', 'number']].drop_duplicates()
trtmtEnv.shape

trtmtEnv.head(10)


,treatment,environment,number
0,B-,"#_{a,O}",1.0
33,B-,#_E,2.0
66,T-,#_E,3.0
103,T-,"#_{a,O}",4.0
137,D-,#_E,5.0
174,D-,"#_{a,O}",6.0
208,C-,#_E,7.0
237,C-,#_a,8.0
270,C-,"#_{o,ŭ}",9.0
304,C-,#_ū,10.0


In [38]:
# add id as index
trtmtEnv.rename(columns={'number': 'trtmt_env_id'}, inplace=True)
trtmtEnv.set_index('trtmt_env_id', inplace=True)

#### parse original excel for ref words and sound changes

In [39]:
import json
with open(r"overview_sound_changes_filtered_per_sheet.json", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
rows = []
for key, value in data.items():
    id_part, treatment_part = key.split(" ", 1)

    if id_part.isdigit():
        id_part = float(id_part)
    else:
        # strip leading 0
        id_part = float(id_part.lstrip("0"))
    rows.append({
        "trtmt_env_id": id_part,
        "treatment": treatment_part,
        "sound_changes": value,
    })

df = pd.DataFrame(rows, columns=["trtmt_env_id", "treatment", "sound_changes"])
df["trtmt_env_id"] = df["trtmt_env_id"].astype(str)
df["treatment"] = df["treatment"].astype(str)

In [41]:
# join to trtmt Env
final_df = pd.merge(trtmtEnv, df, on=["trtmt_env_id", "treatment"], how = 'outer')

In [45]:
final_df.sort_values('trtmt_env_id').head(20)

,trtmt_env_id,treatment,environment,sound_changes
0,016.5,M-,NaN,{'m // #_': ['typically /m/']}
1,031a,W-,NaN,"{'031a: w / ? / #_E': ['typically /ɡ/', '/ɡw/ ..."
2,031b,W-,NaN,"{'031b: w / ? / #_{a,O}': ['typically /ɡw/', '..."
3,049a,SC-,NaN,"{'049a₁: sk / ? / #_E': ['/esk/ in POR, ESP, C..."
4,049b,SC-,NaN,"{'049b₁: sk / ? / #_E': ['/s/ in POR, CVB, AND..."
5,083.5,-FJ-,NaN,{'083.5: fj / ? / V_V': ['typically either /fj...
6,1.0,B-,"#_{a,O}","{'b // #_{a,O}': ['typically /b/']}"
7,10.0,C-,#_ū,{'k // #_ū': ['typically /k/']}
8,100.0,-FF-,V_V,{'ff / ? / V_V': ['typically degeminated; the ...
9,101.0,-SS-,V_E,{'101a: ss / ? / V_E': ['typically palatalized...


In [4]:
qdfh = pd.ExcelFile('QDFH.xlsx')

sheet_names = qdfh.sheet_names
print(list(sheet_names))
data_sheets = sheet_names[1:-4]

['Languages Overview', '001 B-', '002 B-', '003 T-', '004 T-', '005 D-', '006 D-', '007 C-', '008 C-', '009 C-', '010 C-', '011 G-', '012 G-', '013 G-', '014 G-', '015 N-', '016 N-', '016.5 M-', '017 F-', '018 F-', '019 V-', '020 V-', '021 S-', '022 S-', '023 S-', '024 H-', '025 R-', '026 L-', '027 L-', '028 J-', '029 J-', '030 J-', '031a W-', '031b W-', '032 PL-', '033 BR-', '034 BL-', '035 TR-', '036 DR-', '037 CR-', '038 CL-', '039 QU-', '040 QU-', '041 QU-', '042 QU-', '043 GL-', '044 FR-', '045 FL-', '046 SP-', '047 ST-', '048 ST-', '049a SC-', '049b SC-', '050 SC-', '051 STR-', '052 SCR-', '053 -P-', '054 -B-', '055 -T-', '056 -T-', '057 -D-', '058 -D-', '059 -C-', '060 -C-', '061 -C-', '062 -G-', '063 -G-', '064 -G-', '065 -M-', '066 -N-', '067 -F-', '068 -V-', '069 -S-', '070 -S-', '071 -R-', '072 -L-', '073 -J-', '074 -J-', '075 -PJ-', '076 -BJ-', '077 -TJ-', '078 -TJ-', '079 -DJ-', '080 -CJ-', '081 -CJ-', '082 -GJ-', '083 -VJ-', '083.5 -FJ-', '084 -SJ-', '085 -MJ-', '086 -NJ-